# Распознавание документов

Задайте пути в первой ячейке и выполните остальные сверху вниз.
Используются Python текущего ядра и уже установленные локальные модели.
Результат: JSON, Markdown, HTML, PDF и, при включении, DOCX.

In [ ]:
from pathlib import Path
from datetime import datetime
import os
import signal
import subprocess
import sys

PROJECT_ROOT = Path.cwd().resolve()
INPUTS = [PROJECT_ROOT / "data" / "input"]  # Файлы и/или папки
OUTPUT_ROOT = PROJECT_ROOT / "reports" / "notebook-runs"
CONFIG_FILE = PROJECT_ROOT / "configs" / "nemotron-review.yaml"

DOCLING_MODELS = None  # Локальный кэш Docling; можно указать свой каталог
import rapidocr
RAPIDOCR_MODELS = Path(rapidocr.__file__).parent / "models"
RUSSIAN_MODEL = PROJECT_ROOT / "models" / "ppocrv6-medium-ru"
NEMOTRON_MODEL = PROJECT_ROOT / "models" / "nemotron-3.5-parse"
NEMOTRON_ENABLED = True
NEMOTRON_DEVICE = "cpu"  # cpu, cuda, auto
CPU_THREADS = 6
EXPORT_DOCX = False
EXPORT_SEARCHABLE_PDF = False

In [ ]:
def project_path(value):
    path = Path(value).expanduser()
    return (PROJECT_ROOT / path).resolve() if not path.is_absolute() else path.resolve()

PROJECT_ROOT = project_path(PROJECT_ROOT)
if not (PROJECT_ROOT / "src" / "docpipe" / "cli.py").is_file():
    raise FileNotFoundError("Укажите PROJECT_ROOT — папку с исходниками проекта")
CONFIG_FILE = project_path(CONFIG_FILE)
DOCLING_MODELS, RAPIDOCR_MODELS, RUSSIAN_MODEL, NEMOTRON_MODEL = (
    project_path(path) if path is not None else None
    for path in (DOCLING_MODELS, RAPIDOCR_MODELS, RUSSIAN_MODEL, NEMOTRON_MODEL)
)
required = [CONFIG_FILE,
    RUSSIAN_MODEL / "model.onnx", RUSSIAN_MODEL / "rec_keys.txt"]
required += [RAPIDOCR_MODELS / name for name in (
    "ch_PP-OCRv5_det_mobile.onnx", "ch_ppocr_mobile_v2.0_cls_mobile.onnx",
    "eslav_PP-OCRv5_rec_mobile.onnx", "latin_PP-OCRv5_rec_mobile.onnx")]
if NEMOTRON_ENABLED:
    required += [NEMOTRON_MODEL / name for name in (
        "model.safetensors", "config.json", "hf_nemotron_parse_modeling.py",
        "hf_nemotron_parse_processor.py", "nemotron_parse/__init__.py")]
if DOCLING_MODELS is not None:
    required.append(DOCLING_MODELS)
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Не найдены локальные файлы/каталоги: " + "; ".join(missing))

env = {key: value for key, value in os.environ.items() if not key.startswith("DOCPIPE_")}
temp_dir = PROJECT_ROOT / "reports" / "notebook-tmp"
temp_dir.mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONPATH": str(PROJECT_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", ""),
    "PYTHONIOENCODING": "utf-8", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "DOCPIPE_RAPIDOCR_MODEL_DIR": str(RAPIDOCR_MODELS),
    "DOCPIPE_RU_REC_MODEL_DIR": str(RUSSIAN_MODEL),
    "DOCPIPE_OFFLINE": "true", "DOCPIPE_REVIEW__PROVIDER": "nemotron",
    "DOCPIPE_REVIEW__LLM_ENABLED": str(NEMOTRON_ENABLED).lower(),
    "DOCPIPE_REVIEW__NEMOTRON_MODEL": str(NEMOTRON_MODEL),
    "DOCPIPE_REVIEW__NEMOTRON_DEVICE": NEMOTRON_DEVICE,
    "DOCPIPE_RENDER__DOCX_ENABLED": str(EXPORT_DOCX).lower(),
    "OMP_NUM_THREADS": str(CPU_THREADS), "MKL_NUM_THREADS": str(CPU_THREADS),
    "TEMP": str(temp_dir), "TMP": str(temp_dir), "TMPDIR": str(temp_dir),
})
if DOCLING_MODELS is not None:
    env["DOCLING_ARTIFACTS_PATH"] = str(DOCLING_MODELS)
print("Python:", sys.executable)

In [ ]:
extensions = {".pdf", ".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}
files = set()
for value in INPUTS:
    path = project_path(value)
    if not path.exists():
        raise FileNotFoundError(path)
    files.update(file for file in (path.iterdir() if path.is_dir() else [path])
        if file.is_file() and file.suffix.lower() in extensions)
if not files:
    raise ValueError("Нет документов для распознавания: проверьте INPUTS")

run_dir = project_path(OUTPUT_ROOT) / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
requested = ["md", "html", "pdf-flow", "pdf-positional"]
if EXPORT_SEARCHABLE_PDF:
    requested.append("pdf-searchable")
for index, source in enumerate(sorted(files), start=1):
    out = run_dir / f"{index:03d}_{source.stem}"
    command = [sys.executable, "-m", "docpipe", "parse", str(source),
        "--out", str(out), "--config", str(CONFIG_FILE), "--outputs", *requested]
    options = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if os.name == "nt" else {"start_new_session": True}
    print("Распознавание:", source, flush=True)
    process = subprocess.Popen(command, cwd=PROJECT_ROOT, env=env, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", **options)
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError(f"Не удалось обработать {source}")
    except BaseException:
        if process.poll() is None:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(process.pid), "/T", "/F"], capture_output=True)
            else:
                os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        raise
    finally:
        process.stdout.close()
    print("Результат:", out)